# 02 Incremental load
Reads the watermark, pulls rows with `updated_at > watermark` from the source, appends them to Bronze and stages them for the Silver MERGE.

**Known limits:** a watermark on `updated_at` cannot see hard deletes in the source, and re-running this task before `07_update_metadata` appends the same rows to Bronze again (Silver stays correct because the MERGE keeps one row per key).

In [ ]:
def jdbc_options():
    """Read connection details from a Databricks secret scope (never hard-code credentials)."""
    scope = "sales_pipeline"
    host = dbutils.secrets.get(scope, "pg_host")
    database = dbutils.secrets.get(scope, "pg_database")
    return {
        "url": f"jdbc:postgresql://{host}:5432/{database}?sslmode=require",
        "user": dbutils.secrets.get(scope, "pg_user"),
        "password": dbutils.secrets.get(scope, "pg_password"),
        "driver": "org.postgresql.Driver",
    }

In [ ]:
watermark = spark.sql("""
SELECT watermark FROM metadata.pipeline_metadata WHERE pipeline_name = 'sales_pipeline'
""").collect()[0][0]
print(f"Current watermark: {watermark}")

In [ ]:
from pyspark.sql import functions as F

df_incremental = (
    spark.read.format("jdbc")
    .options(**jdbc_options())
    .option("query", f"SELECT * FROM sales WHERE updated_at > '{watermark}'")
    .load()
)
record_count = df_incremental.count()
print(f"Rows fetched from source: {record_count}")

In [ ]:
if record_count == 0:
    print("No new records found.")
    if spark.catalog.tableExists("staging.sales_incremental"):
        spark.sql("TRUNCATE TABLE staging.sales_incremental")
else:
    df_incremental = df_incremental.withColumn("ingestion_time", F.current_timestamp())
    df_incremental.write.format("delta").mode("append").saveAsTable("bronze.sales")
    df_incremental.write.format("delta").mode("overwrite").saveAsTable("staging.sales_incremental")
    print("Incremental load successful")